# Optional MLflow Tracking and Model Registry quickstart

This notebook is an additive ML example. It uses the standalone MLflow service and leaves the core lakehouse resources unchanged. Start the lakehouse with Jupyter, then start the MLflow Compose add-on as documented in the repository README.

The first cell installs the MLflow client and scikit-learn into this running Jupyter container. Re-run it after recreating the container.

In [ ]:
%pip install mlflow==3.17.2 scikit-learn

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Containers on lakedemo reach the standalone tracking server by service name.
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("lakehouse-ml-quickstart")

X, y = make_regression(n_samples=300, n_features=4, noise=12, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model = LinearRegression()
model.fit(X_train, y_train)
rmse = mean_squared_error(y_test, model.predict(X_test)) ** 0.5

with mlflow.start_run() as run:
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("n_features", X.shape[1])
    mlflow.log_metric("rmse", rmse)
    mlflow.sklearn.log_model(
        sk_model=model,
        name="model",
        registered_model_name="lakehouse-demo-regressor",
    )
    print(f"Logged run {run.info.run_id}; RMSE={rmse:.3f}")

Open the MLflow UI at [http://localhost:5000](http://localhost:5000). The experiment appears under **Experiments**; the registered model appears under **Models**. The run metadata is in MLflow's dedicated PostgreSQL volume and model files are in its dedicated artifact volume.